In [1]:
import matplotlib.pyplot as plt, numpy as np, seaborn as sns, scipy.stats as stats, pandas as pd, os, glob
import ast
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

### 1. preprocess old versions

In [2]:
# create concatenated df
df_old = pd.DataFrame()

for subj_file in glob.glob('../../results/psychopy/asymmetry*.csv'):
    df_subj = pd.read_csv(subj_file)[:240]
    df_old = pd.concat([df_old, df_subj], ignore_index=True)

df_old.to_csv('../../results/psychopy/all_subjs_v2.csv', index=False)
df_old

,thisN,thisTrialN,thisRepN,blockN,run,condition,trial_key,stim_file_pos,stim_pos,noise_pos,...,subj,difficulty,sess_type,date,expName,psychopyVersion,frameRate,expStart,Unnamed: 119,Unnamed: 112
0,0.0,0.0,0.0,1.0,1.0,baseline,10.0,1.1,-0.18,1.0,...,202509.0,patients,A,2025-05-23_14h44.58.284,asymmetry_final,2024.2.4,60.0,2025-05-23 14h45.10.782086 -0600,NaN,NaN
1,1.0,1.0,0.0,1.0,1.0,baseline,7.0,0.7,-0.26,1.0,...,202509.0,patients,A,2025-05-23_14h44.58.284,asymmetry_final,2024.2.4,60.0,2025-05-23 14h45.10.782086 -0600,NaN,NaN
2,2.0,2.0,0.0,1.0,1.0,baseline,37.0,3.7,0.34,3.0,...,202509.0,patients,A,2025-05-23_14h44.58.284,asymmetry_final,2024.2.4,60.0,2025-05-23 14h45.10.782086 -0600,NaN,NaN
3,3.0,3.0,0.0,1.0,1.0,baseline,14.0,1.5,-0.10,1.0,...,202509.0,patients,A,2025-05-23_14h44.58.284,asymmetry_final,2024.2.4,60.0,2025-05-23 14h45.10.782086 -0600,NaN,NaN
4,4.0,4.0,0.0,1.0,1.0,baseline,8.0,0.9,-0.22,1.0,...,202509.0,patients,A,2025-05-23_14h44.58.284,asymmetry_final,2024.2.4,60.0,2025-05-23 14h45.10.782086 -0600,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1435,35.0,35.0,0.0,6.0,2.0,flat_comp,236.0,3.9,0.38,1.0,...,202518.0,patients,D,2025-10-04_16h05.12.182,asymmetry_final,2024.2.4,60.0,2025-10-04 16h05.25.163659 -0600,NaN,NaN
1436,36.0,36.0,0.0,6.0,2.0,flat_comp,226.0,3.3,0.26,1.0,...,202518.0,patients,D,2025-10-04_16h05.12.182,asymmetry_final,2024.2.4,60.0,2025-10-04 16h05.25.163659 -0600,NaN,NaN
1437,37.0,37.0,0.0,6.0,2.0,flat_comp,223.0,3.1,0.22,1.0,...,202518.0,patients,D,2025-10-04_16h05.12.182,asymmetry_final,2024.2.4,60.0,2025-10-04 16h05.25.163659 -0600,NaN,NaN
1438,38.0,38.0,0.0,6.0,2.0,flat_comp,218.0,2.9,0.18,3.0,...,202518.0,patients,D,2025-10-04_16h05.12.182,asymmetry_final,2024.2.4,60.0,2025-10-04 16h05.25.163659 -0600,NaN,NaN


In [3]:
df_old['direction'] = df_old['shape_order'].map({'curv_flat': 1, 'flat_curv': -1})
df_old['true_div'] = np.where( df_old['direction'] == -1, -df_old['div_pos'], df_old['div_pos'] ) 

df_old[['shape_order', 'direction', 'true_div']].head(5)

,shape_order,direction,true_div
0,curv_flat,1,0.0
1,flat_curv,-1,-0.0
2,flat_curv,-1,-0.0
3,curv_flat,1,0.0
4,curv_flat,1,0.0


valence flipping entire sessions because we didnt counterbalance halfway

In [4]:
# Assert valence mapping for each session type
df_sessions_AC = df_old[df_old['sess_type'].isin(['A', 'C'])]
assert (df_sessions_AC[df_sessions_AC['shape'] == 'curv']['valence'] == 'pen').all()
assert (df_sessions_AC[df_sessions_AC['shape'] == 'flat']['valence'] == 'rew').all()

df_sessions_BD = df_old[df_old['sess_type'].isin(['B', 'D'])]
assert (df_sessions_BD[df_sessions_BD['shape'] == 'curv']['valence'] == 'rew').all()
assert (df_sessions_BD[df_sessions_BD['shape'] == 'flat']['valence'] == 'pen').all()

# Create valence_flipped column: 1 for A/C, -1 for B/D
df_old['valence_flipped'] = np.where(df_old['sess_type'].isin(['A', 'C']), 1, -1)

### 2. preprocess new versions

In [5]:
# create concatenated df
df_new = pd.DataFrame()

for subj_file in glob.glob('../../results/psychopy/asymmetry_v2*.csv'):
    df_subj = pd.read_csv(subj_file)[:240]
    df_new = pd.concat([df_new, df_subj], ignore_index=True)

df_new

""


### 3. combine dfs and global preprocess

In [6]:
df_subjs = pd.concat([df_old, df_new], ignore_index=True)
print(df_subjs.shape)
df_subjs.head(5)

(1440, 124)


,thisN,thisTrialN,thisRepN,blockN,run,condition,trial_key,stim_file_pos,stim_pos,noise_pos,...,date,expName,psychopyVersion,frameRate,expStart,Unnamed: 119,Unnamed: 112,direction,true_div,valence_flipped
0,0.0,0.0,0.0,1.0,1.0,baseline,10.0,1.1,-0.18,1.0,...,2025-05-23_14h44.58.284,asymmetry_final,2024.2.4,60.0,2025-05-23 14h45.10.782086 -0600,NaN,NaN,1,0.0,1
1,1.0,1.0,0.0,1.0,1.0,baseline,7.0,0.7,-0.26,1.0,...,2025-05-23_14h44.58.284,asymmetry_final,2024.2.4,60.0,2025-05-23 14h45.10.782086 -0600,NaN,NaN,-1,-0.0,1
2,2.0,2.0,0.0,1.0,1.0,baseline,37.0,3.7,0.34,3.0,...,2025-05-23_14h44.58.284,asymmetry_final,2024.2.4,60.0,2025-05-23 14h45.10.782086 -0600,NaN,NaN,-1,-0.0,1
3,3.0,3.0,0.0,1.0,1.0,baseline,14.0,1.5,-0.10,1.0,...,2025-05-23_14h44.58.284,asymmetry_final,2024.2.4,60.0,2025-05-23 14h45.10.782086 -0600,NaN,NaN,1,0.0,1
4,4.0,4.0,0.0,1.0,1.0,baseline,8.0,0.9,-0.22,1.0,...,2025-05-23_14h44.58.284,asymmetry_final,2024.2.4,60.0,2025-05-23 14h45.10.782086 -0600,NaN,NaN,1,0.0,1


In [7]:
# convert string to list, and store target_response
for column in ['positions']:
    if type(df_subjs[column][0]) == str:
        df_subjs.loc[:, column] = df_subjs[column].apply(ast.literal_eval)       
df_subjs['target_response'] = df_subjs['positions'].apply(lambda x: x[-1])

# true_response column flips target_response when direction = -1 (flat_curv)
df_subjs['true_response'] = np.where( df_subjs['direction'] == -1, -df_subjs['target_response'], df_subjs['target_response'] ) 

df_subjs[['shape_order', 'direction', 'target_response', 'true_response']].head(5)

,shape_order,direction,target_response,true_response
0,curv_flat,1,0.064,0.064
1,flat_curv,-1,0.100,-0.100
2,flat_curv,-1,-0.136,0.136
3,curv_flat,1,-0.136,-0.136
4,curv_flat,1,0.056,0.056


In [8]:
# rename certain columns
df_subjs = df_subjs.rename(columns={'condition':'shape_condition',# 'valence':'valence_old',
                                    'stim_pos':'true_stim', 'target_pos':'target_stim', 
                                    'div_pos':'target_div'})

df_subjs[['shape_order', 'direction',
          'true_stim', 'target_stim',
          'true_div', 'target_div',
          'true_response', 'target_response']].head(5)


,shape_order,direction,true_stim,target_stim,true_div,target_div,true_response,target_response
0,curv_flat,1,-0.18,-0.18,0.0,0.0,0.064,0.064
1,flat_curv,-1,-0.26,0.26,-0.0,0.0,-0.100,0.100
2,flat_curv,-1,0.34,-0.34,-0.0,0.0,0.136,-0.136
3,curv_flat,1,-0.10,-0.10,0.0,0.0,-0.136,-0.136
4,curv_flat,1,-0.22,-0.22,0.0,0.0,0.056,0.056


Important: flip positions based on valence
* when curvy = loss (& flat = gain), all is fine
* when curvy = gain (& flat = loss), need to flip positions

In [12]:
# find indices and flip
df_subjs['valence_flipped'] = (
    ((df_subjs['shape'] == 'curv') & (df_subjs['valence'] == 'rew')) |
    ((df_subjs['shape'] == 'flat') & (df_subjs['valence'] == 'pen'))
)

# reminder that below wont be true because of older version files
# assert (df_subjs['valence_flipped'] == (df_subjs['blockN'] >= 4)).all()

df_subjs['valence_stim'] = np.where(df_subjs['valence_flipped'], -df_subjs['true_stim'], df_subjs['true_stim'])
df_subjs['valence_div'] = np.where(df_subjs['valence_flipped'], -df_subjs['true_div'], df_subjs['true_div'])
df_subjs['valence_response'] = np.where(df_subjs['valence_flipped'], -df_subjs['true_response'], df_subjs['true_response'])

df_subjs[['blockN', 'shape', 'valence',
          'true_stim', 'valence_stim',
          'true_div', 'valence_div',
          'true_response', 'valence_response']].head(5)

,blockN,shape,valence,true_stim,valence_stim,true_div,valence_div,true_response,valence_response
0,1.0,curv,pen,-0.18,-0.18,0.0,0.0,0.064,0.064
1,1.0,curv,pen,-0.26,-0.26,-0.0,-0.0,-0.100,-0.100
2,1.0,flat,rew,0.34,0.34,-0.0,-0.0,0.136,0.136
3,1.0,curv,pen,-0.10,-0.10,0.0,0.0,-0.136,-0.136
4,1.0,curv,pen,-0.22,-0.22,0.0,0.0,0.056,0.056


In [10]:
# error trials
df_subjs['invalid'] = df_subjs['trials.slider_resp.rt'].isna()
df_subjs['incomplete'] = df_subjs['trials.submit_resp.keys'].isna()
df_subjs['incorrect'] = (df_subjs['correct']==0) & ~df_subjs['invalid'] & ~df_subjs['incomplete']
df_subjs[['correct', 'invalid', 'incomplete', 'incorrect']].head(5)

,correct,invalid,incomplete,incorrect
0,0.0,False,False,True
1,1.0,False,False,False
2,0.0,True,False,False
3,1.0,False,False,False
4,0.0,False,False,True


### global preprocessing, misc

In [11]:
# category stuff
df_subjs['true_class'] = np.where( df_subjs['valence'] == 'rew', 1, 0 ) 
df_subjs['pred_class'] = (df_subjs['div_pos_aligned'] < df_subjs['resp_aligned']).astype(int)
df_subjs['err_type'] = df_subjs['pred_class'] - df_subjs['true_class']

# boundary stuff
df_subjs['uncertainty'] = (df_subjs['stim_pos'] - df_subjs['div_pos_aligned']).abs() < 0.2
df_subjs['stim_aligned_to_div'] = (df_subjs['stim_pos'] - df_subjs['div_pos_aligned']).round(3)
df_subjs['resp_aligned_to_div'] = (df_subjs['resp_aligned'] - df_subjs['div_pos_aligned']).round(3)
# if loss,  if stim is on penalty side of div, pos if on reward side
df_subjs['stim_aligned_to_cntxt'] = np.where( df_subjs['true_class'] == 1, df_subjs['stim_aligned_to_div'], - df_subjs['stim_aligned_to_div'] )
df_subjs['resp_aligned_to_cntxt'] = np.where( df_subjs['true_class'] == 1, df_subjs['resp_aligned_to_div'], - df_subjs['resp_aligned_to_div'] )

# rank within a block for each subj/cond
grp = ['subj', 'condition']#, 'blockN']
# build both ranks, assign once, then copy to defragment
df_subjs = df_subjs.assign(
    stim_ranks = df_subjs.groupby(grp)['stim_pos'].transform('rank'),
    resp_ranks = df_subjs.groupby(grp)['resp_aligned'].transform('rank'),
).copy()
max_rank = df_subjs['stim_ranks'].max()

# boundary-aligned rank
df_subjs['stim_ranks_aligned'] = df_subjs['stim_ranks'] - max_rank/2
df_subjs['resp_ranks_aligned'] = df_subjs['resp_ranks'] - max_rank/2
print(max_rank/2)

# create column block_name which concats condition and run
df_subjs['block_name'] = df_subjs['condition'] + '_' + df_subjs['run'].astype(str)


KeyError: 'div_pos_aligned'

### checks

In [ ]:
# sort by subj
df_subjs = df_subjs.sort_values(by=['subj']).reset_index(drop=True)
df_subjs.to_csv('../../results/psychopy/all_subjs_clean.csv', index=False)

# asserts
print(df_subjs['outcome'].value_counts(), '\n')
print(df_subjs['correct'].value_counts(), '\n')
print(df_subjs['uncertainty'].value_counts(), '\n')
print(df_subjs.shape, '\n')
print(len(df_subjs))
assert len(df_subjs) == 240 * len(raw_pt_IDs), "Total trials do not match expected number"

disp_cols = ['subj', 'sess_type', 'block_name', 'sess_flip', 'shape_order', 'dir_flip', 'target_pos', 'shape', 'valence', 'true_class',
             'div_pos', 'div_pos_aligned', 'stim_pos', 'stim_pos', 'chosen_pos', 'chosen_pos_aligned', 'pred_class', 'err_type',
             'signed_err', 'unsigned_err']

df_subjs[(df_subjs['sess_type'] == 'B') &
         (df_subjs['condition'] == 'baseline') &
         (df_subjs['err_type'] != 0) &
         (df_subjs['sess_flip'] != df_subjs['dir_flip'])
        ][disp_cols][:20]

In [ ]:
# use groupby.head(1) to keep the first trial for each subject
df_first_per_subj = df_subjs.groupby('subj').head(1)[['subj', 'sess_type']].reset_index(drop=True)
df_first_per_subj